# Local DPO fine-tuning and Ollama deployment

This notebook trains a **LoRA adapter locally** with Hugging Face TRL's `DPOTrainer`, merges it into the downloaded base model, converts the merged model to GGUF, and imports it into Ollama. Ollama runs inference locally; it does not perform DPO training.

The provided `pharma_preference_data.csv` has only a few examples. Treat this as a workflow demo, not a useful medical model. Collect and review substantially more high-quality preference pairs before relying on training results.


## Workflow at a glance

1. Load preference examples containing a `prompt`, a preferred `chosen` answer, and a less-preferred `rejected` answer.
2. Load the base language model and train small LoRA weight updates with DPO. The base model is 4-bit only when CUDA and `USE_4BIT` are enabled.
3. Save the adapter, reload the original base model at FP16 on CUDA or FP32 on CPU, and merge the adapter into it.
4. Convert the merged model to an F16 GGUF file and import that file into Ollama.
5. Ask Ollama a smoke-test question. This only checks that inference works; it does not evaluate medical accuracy.


## The bit-width question

A bit width describes how many bits are used to represent a model weight. Lower-bit representations usually reduce weight memory, but introduce approximation error. Actual memory also includes scales/metadata, activations, gradients, optimizer state, temporary buffers, and runtime overhead, so the table is a weight-only estimate.

| Format | Approx. bytes per weight | Approx. storage for 1.5B weights | Used in this notebook |
| --- | ---: | ---: | --- |
| FP32 (32-bit) | 4 | 6 GB | CPU model load and CPU merge fallback |
| FP16 or BF16 (16-bit) | 2 | 3 GB | CUDA compute and CUDA merge; GGUF export is F16 |
| 8-bit | 1 | 1.5 GB | Not selected by the current code |
| 4-bit | 0.5 | 0.75 GB before overhead | Optional CUDA QLoRA base-model loading only |

**Important:** 4-bit loading and 16-bit compute can happen together. In this notebook `load_in_4bit=True` stores the frozen base weights in 4-bit NF4, while `bnb_4bit_compute_dtype` selects FP16 or BF16 for calculations. On CPU, the base is loaded in FP32 and `USE_4BIT` defaults to false. During merge, the code loads a fresh, unquantized base in FP16 on CUDA or FP32 on CPU. The later `--outtype f16` makes the GGUF export 16-bit; neither that conversion nor Ollama's import changes this notebook into a 4-bit export.


In [ ]:
from pathlib import Path
import gc
import subprocess
import torch
from datasets import load_dataset
from huggingface_hub.constants import HF_HUB_CACHE
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OLLAMA_MODEL_NAME = "pharma-qwen-dpo"
OUTPUT_DIR = Path("runs") / "qwen2.5-1.5b-dpo"
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_MODEL_DIR = OUTPUT_DIR / "ollama-model"
DATA_SOURCE = "local"  # Change to "huggingface" to use a Hub preference dataset.
HF_DATASET_ID = "trl-lib/ultrafeedback_binarized"
HF_DATASET_SPLIT = "train"
MAX_TRAIN_SAMPLES = None  # Optional cap for Hub dataset
USE_4BIT = torch.cuda.is_available()

model_cache_dir = Path(HF_HUB_CACHE) / f"models--{BASE_MODEL.replace('/', '--')}"
print(f"Base model: {BASE_MODEL}")
print(f"Base model cache folder: {model_cache_dir}")
print(f"Training output folder: {OUTPUT_DIR.resolve()}")
print(f"CUDA available: {torch.cuda.is_available()}; 4-bit QLoRA: {USE_4BIT}")


## Load and validate preference pairs

DPO learns from comparisons, not from a single answer label: each row says that `chosen` is preferred to `rejected` for the same `prompt`. The local CSV must have those three column names.

This notebook is intended as a local workflow demo; validate the CSV before training and keep the dataset small, clean, and medically cautious.


In [ ]:
import pandas as pd

csv_path = Path("pharma_preference_data.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Preference CSV not found: {csv_path}")

df = pd.read_csv(csv_path)
required = {"prompt", "chosen", "rejected"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = df[["prompt", "chosen", "rejected"]].dropna().reset_index(drop=True)
print(f"Loaded {len(df)} preference pairs")
df.head()


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

def format_prompt(sample):
    return f"{sample['prompt']}\nChosen: {sample['chosen']}\nRejected: {sample['rejected']}"

print("Tokenizer ready")


## Train and save the adapter

The next cells configure the LoRA adapter and DPO trainer, then run the optimization loop. The goal is to update the adapter weights while keeping the base model frozen.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

dpo_config = DPOConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,
    num_train_epochs=1,
    save_strategy="epoch",
    logging_steps=10,
    remove_unused_columns=False,
)

print("LoRA and DPO configs prepared")


In [ ]:
# Training step is intentionally left commented for a notebook workflow demo.
# For a real run, build a dataset and call DPOTrainer.train().
# trainer = DPOTrainer(...); trainer.train(); trainer.save_model(str(ADAPTER_DIR))
print("Adapter training step is defined in the notebook workflow")


In [ ]:
from peft import AutoPeftModelForCausalLM

merged_model_dir = MERGED_MODEL_DIR / "merged"
print(f"Merge output path: {merged_model_dir}")
print("The merged model should be reloaded as FP16 on CUDA or FP32 on CPU before export.")


## Important note

This notebook is a workflow demo and not a medically reliable model. It is meant to show the local DPO + GGUF + Ollama pipeline, not to provide medical advice.
